# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the [`mlcroissant`](https://github.com/mlcommons/croissant) library to load, inspect, and process the FAIR<sup>2</sup> dataset defined by a Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant library
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Display basic metadata
meta = dataset.metadata
print(f"Dataset: {meta.name}\nDescription: {meta.description}\nID: {meta.id}\nVersion: {getattr(meta, 'version', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The Croissant metadata contains one or more record sets (`cr:RecordSet`).
We will enumerate them below along with their fields and columns, all referenced by their `@id`.

In [ ]:
# Helper function to get list of objects as lists
def ensure_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    else:
        return [x]

# Print information for all record sets
record_sets = getattr(meta, 'recordSet', [])
if not record_sets:
    print('No record sets found! Please check the schema for proper record set definition.')
else:
    print("Available record sets and fields (by @id):\n")
    for recset in ensure_list(record_sets):
        # recset is a mlc.RecordSet object
        print(f"Record set: {recset.id} (name: {getattr(recset,'name', 'N/A')})")
        fields = getattr(recset, 'field', [])
        if not fields:
            print("  No fields defined in this record set.")
        else:
            for f in ensure_list(fields):
                print(f"   - Field: {f.id}  (name: {getattr(f, 'name', 'N/A')}) | dataType: {getattr(f, 'dataType', 'N/A')}")
                # If columns exist, print them
                cols = getattr(f, 'column', [])
                for col in ensure_list(cols):
                    print(f"       - Column: {col.id}  (name: {getattr(col, 'name', 'N/A')})")
        print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Pick all record set @id's for extraction by inspecting metadata

# List of record set @id's
all_record_sets = []
for recset in ensure_list(getattr(meta, 'recordSet', [])):
    all_record_sets.append(recset.id)

print("Extracting record sets:", all_record_sets)

dataframes = {}
for recset_id in all_record_sets:
    print(f"Loading records from record set: {recset_id}")
    records = list(dataset.records(record_set=recset_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[recset_id] = df
    # Show minimal preview if data exists
    if not df.empty:
        print(f"Columns for {recset_id}: {df.columns.tolist()}")
        display(df.head(2))
    else:
        print(f"No data loaded for record set: {recset_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming distributions, or grouping data by key attributes to prepare for further analysis.


> _**Note:** Modify field `@id`s appropriately, based on the record set contents. In this example, we will try to select a numeric field like `"coverage_pct"` or `"mw"` if available._

In [ ]:
# -------
# Select a record set and numeric field for demo

# By default, pick the first non-empty record set
selected_record_set = None
for rec_id, df in dataframes.items():
    if not df.empty:
        selected_record_set = rec_id
        break

if selected_record_set is None:
    print("No data is loaded! Cannot proceed with EDA.")
else:
    df = dataframes[selected_record_set]

    # Try to select a numeric field
    candidate_numeric_fields = ['mw', 'coverage_pct', 'peptide_count', 'normalized_abundance']
    # Use names as they appear in DF columns
    numeric_field = None
    for field in candidate_numeric_fields:
        if field in df.columns:
            numeric_field = field
            break
    if numeric_field is None:
        # Just pick the first numeric-type column (float or int)
        for col in df.select_dtypes(include=['float', 'int']).columns:
            numeric_field = col
            break
    if numeric_field is None:
        print("No numeric field found for EDA.")
    else:
        print(f"Using numeric field '{numeric_field}' for EDA.")

        # Set a threshold, e.g. 10
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold} (showing top 5):")
        display(filtered_df.head(5))

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} (showing first 5):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a field, e.g. 'description', if present
        group_field = None
        for candidate in ['description', 'accession', 'sample']:
            if candidate in df.columns:
                group_field = candidate
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(f"mean_{numeric_field}")
            print(f"Grouped data by '{group_field}' (showing 5 groups):")
            display(grouped_df.head())
        else:
            print("No suitable group field found in columns for grouping analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set and numeric_field and not filtered_df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True, color='royalblue')
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field} (filtered > {threshold})")
    plt.show()

    # Optionally, show relationship with another variable if present
    another_numeric = [col for col in filtered_df.columns if col != numeric_field and pd.api.types.is_numeric_dtype(filtered_df[col])]
    if another_numeric:
        plt.figure(figsize=(6,6))
        sns.scatterplot(data=filtered_df, x=numeric_field, y=another_numeric[0])
        plt.xlabel(numeric_field)
        plt.ylabel(another_numeric[0])
        plt.title(f"{numeric_field} vs {another_numeric[0]}")
        plt.show()
else:
    print('Not enough data for plotting. Check earlier cells to ensure data is loaded and numeric_field is found.')

## 6. Conclusion
This notebook used the `mlcroissant` library to load, inspect, and process the FAIR<sup>2</sup> mass spectrometry protein dataset. We explored available record sets and fields, loaded the data into pandas DataFrames with columns referenced by their `@id`, filtered and normalized a numeric field, and visualized the results.

**Key next steps:**
- Investigate other fields of interest, such as peptide modifications, or sample descriptions.
- Join across record sets using shared identifiers if relevant.
- Apply statistical or machine learning analysis as required for your use case.